In [1]:
import logging
import os
import numpy as np

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

from napistu_torch.load.constants import FM_DEFS, SCGPT_DEFS
from napistu_torch.load.foundation_models import FoundationModel
from napistu_torch.load.foundation_model_etl import process_scgpt

In [2]:
# Configuration
DATA_DIR = "data"
OUTPUT_DIR = "output"

MODEL_PATH = os.path.join(DATA_DIR, "scGPT_bc")
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
process_scgpt(MODEL_PATH, OUTPUT_DIR)

INFO:datasets:PyTorch version 2.3.0 available.
INFO:datasets:JAX version 0.7.2 available.
INFO:napistu_torch.load.foundation_model_etl:Extracting: scGPT
INFO:napistu_torch.load.foundation_model_etl:
1. Downloading/loading gene annotations...
INFO:napistu_torch.load.foundation_model_etl:   Loaded 60664 gene annotations
INFO:napistu_torch.load.foundation_model_etl:2. Loading scGPT model...


Resume model from data/scGPT_bc/best_model.pt, the model args will override the config data/scGPT_bc/args.json.
Loading params encoder.embedding.weight with shape torch.Size([60697, 512])
Loading params encoder.enc_norm.weight with shape torch.Size([512])
Loading params encoder.enc_norm.bias with shape torch.Size([512])
Loading params value_encoder.linear1.weight with shape torch.Size([512, 1])
Loading params value_encoder.linear1.bias with shape torch.Size([512])
Loading params value_encoder.linear2.weight with shape torch.Size([512, 512])
Loading params value_encoder.linear2.bias with shape torch.Size([512])
Loading params value_encoder.norm.weight with shape torch.Size([512])
Loading params value_encoder.norm.bias with shape torch.Size([512])
Loading params transformer_encoder.layers.0.self_attn.out_proj.weight with shape torch.Size([512, 512])
Loading params transformer_encoder.layers.0.self_attn.out_proj.bias with shape torch.Size([512])
Loading params transformer_encoder.layers.0

INFO:napistu_torch.load.foundation_model_etl:   60664 genes, 12 layers
INFO:napistu_torch.load.foundation_model_etl:3. Extracting weights...
INFO:napistu_torch.load.foundation_model_etl:   Embeddings: (60697, 512)
INFO:napistu_torch.load.foundation_model_etl:   Attention weights: 12 layers × 4 matrices (Q,K,V,O)
INFO:napistu_torch.load.foundation_model_etl:Creating FoundationModel and saving...
INFO:napistu_torch.load.foundation_models:Saving weights to output/scGPT_weights.npz
INFO:napistu_torch.load.foundation_models:Saving metadata to output/scGPT_metadata.json


Loading params transformer_encoder.layers.11.self_attn.out_proj.bias with shape torch.Size([512])
Loading params transformer_encoder.layers.11.linear1.weight with shape torch.Size([512, 512])
Loading params transformer_encoder.layers.11.linear1.bias with shape torch.Size([512])
Loading params transformer_encoder.layers.11.linear2.weight with shape torch.Size([512, 512])
Loading params transformer_encoder.layers.11.linear2.bias with shape torch.Size([512])
Loading params transformer_encoder.layers.11.norm1.weight with shape torch.Size([512])
Loading params transformer_encoder.layers.11.norm1.bias with shape torch.Size([512])
Loading params transformer_encoder.layers.11.norm2.weight with shape torch.Size([512])
Loading params transformer_encoder.layers.11.norm2.bias with shape torch.Size([512])
Loading params decoder.fc.0.weight with shape torch.Size([512, 512])
Loading params decoder.fc.0.bias with shape torch.Size([512])
Loading params decoder.fc.2.weight with shape torch.Size([512, 51

INFO:napistu_torch.load.foundation_models:Successfully saved all results
INFO:napistu_torch.load.foundation_model_etl:Successfully saved all results!


In [4]:
# Load FoundationModel
foundation_model = FoundationModel.load(OUTPUT_DIR, SCGPT_DEFS.MODEL_NAME)

GENES_OF_INTEREST = foundation_model.gene_annotations[FM_DEFS.VOCAB_NAME].sample(20000).tolist()
GENE_MASK = [x in GENES_OF_INTEREST for x in foundation_model.ordered_vocabulary]

# Compute attention on demand using FoundationModelWeights method
# This handles multi-head attention properly
layer_11_attn = foundation_model.weights.compute_attention_from_weights(
    layer_idx=11,
    n_heads=foundation_model.n_heads,
    vocab_mask=np.array(GENE_MASK)
)

INFO:napistu_torch.load.foundation_models:Loading weights (scGPT_weights.npz) and metadata (  scGPT_metadata.json) from output_dir (output)
INFO:napistu_torch.load.foundation_models:Successfully loaded all results
